# Contact forces - analysis

Same falling sphere, but the contact is a continuous spring-damper force (stiffness k, damping
ratio ζ, Coulomb friction) instead of an impulse. The scene, k, ζ and dt ranges are defined in
`main.cpp`. Its results:

- `results/convergence.csv`: one row per (solver, dt), at the reference stiffness;
- `results/stability.csv`: one row per (solver, k, dt), with the stability verdict of `main.cpp`;
- `results/energy_drift.csv`: E(t) for a few dt values.

Run `./scripts/physics_benchmark.sh --scenarios Contact_Forces` to produce them and execute this
notebook.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from analysis_utils import RESULTS, STYLE, expected_orders, export_pdf, plot_solvers, save, shade_from, time_series

from matplotlib.colors import LogNorm

# Stability criterion: with damping nothing can add energy, so a run is unstable when its energy
# exceeds E0 by more than this tolerance (a definition, not a measured value)
ENERGY_GAIN_TOL = 0.01

conv = pd.read_csv(RESULTS / "convergence.csv")
stab = pd.read_csv(RESULTS / "stability.csv")
ede = pd.read_csv(RESULTS / "energy_drift.csv")
print(f"convergence.csv: {len(conv)} rows, stability.csv: {len(stab)} rows, energy_drift.csv: {len(ede)} rows")

## Peak height error vs dt

**How it is made.** After each bounce, the highest sampled height is compared with the same peak in
a reference run (RK4 at the reference timestep `DT_REF` of `main.cpp`); the largest relative error
over all bounces is plotted. Only runs that `main.cpp` flags as stable are shown. Dotted lines:
expected slope for each solver.

**What to expect.** The force is continuous, so in principle each solver shows its own order:
**slopes 1, 2 and 4** for Euler, Verlet and RK4. Two effects can lower the observed orders: the force
switches on and off at contact (it is not smooth there), and the apex is the highest *sampled*
point. The reference is itself an RK4 run, which favours RK4 at its smallest dt.

In [ ]:
data = conv[conv["stable"] == 1]
fig, ax = plt.subplots(figsize=(9, 5))
plot_solvers(ax, data, "dt", "max_height_error")
expected_orders(ax, data, "dt", "max_height_error", {"Euler": 1, "Verlet": 2, "RK4": 4})
ax.set_xlabel("dt  (s)")
ax.set_ylabel("max relative peak height error  (vs reference run)")
ax.set_title("Peak height error vs dt")
ax.legend(ncol=2, fontsize=8)
save(fig, "plot_convergence")

## Stability map

**How it is made.** For each solver, stiffness k and dt: colour = largest energy ratio
`max E(t) / E₀` reached during the run. Crossed cells = unstable runs: with damping nothing can add
energy, so a run is counted unstable when its energy exceeds E₀ by more than `ENERGY_GAIN_TOL`
(set in the first cell). Dashed line: theoretical stability limit of an undamped spring,
`ω·dt = 2` for Euler and Verlet and `ω·dt = 2√2` for RK4, with `ω = √(k/m)` (from the CSV).

**What to expect.** Below the line the energy never exceeds its initial value (ratio 1); above it,
the energy grows. Note that a very large dt can also skip the contact altogether, which looks
stable without being meaningful.

In [ ]:
def edges(values):
    """Cell edges for log-spaced grid values: geometric midpoints, extended at both ends."""
    v = np.sort(values)
    mid = np.sqrt(v[:-1] * v[1:])
    return np.concatenate([[v[0] ** 2 / mid[0]], mid, [v[-1] ** 2 / mid[-1]]])

ks, dts = np.sort(stab["k"].unique()), np.sort(stab["dt"].unique())
norm = LogNorm(vmin=1, vmax=max(stab["max_energy_ratio"].max(), 10))
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True, constrained_layout=True)
for ax, solver in zip(axes, ["Euler", "Verlet", "RK4"]):
    sub = stab[stab["solver"] == solver]
    grid = sub.pivot(index="dt", columns="k", values="max_energy_ratio").reindex(index=dts, columns=ks)
    mesh = ax.pcolormesh(edges(ks), edges(dts), grid.to_numpy(), norm=norm, cmap="RdYlGn_r")
    bad = sub[sub["max_energy_ratio"] > 1 + ENERGY_GAIN_TOL]
    ax.plot(bad["k"], bad["dt"], "kx", ms=8, label="unstable (energy gain)")
    limit = "dt_limit_rk4" if solver == "RK4" else "dt_limit_euler_verlet"
    lim = sub.drop_duplicates("k").sort_values("k")
    ax.plot(lim["k"], lim[limit], "k--", lw=1.2, label="theoretical limit")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(edges(ks)[[0, -1]])
    ax.set_ylim(edges(dts)[[0, -1]])
    ax.set_title(solver)
    ax.set_xlabel("Stiffness k  (N/m)")
axes[0].set_ylabel("dt  (s)")
axes[0].legend(fontsize=8, loc="lower left")
fig.colorbar(mesh, ax=axes, label="max E(t) / E₀")
fig.suptitle("Stability map")
save(fig, "plot_stability_map")

## Stability boundary

**How it is made.** For each solver and stiffness, the largest tested dt below the first unstable dt
(same criterion as the map: filled markers), or the largest tested dt if none is unstable (hollow
markers: no instability on the grid). Lines: theoretical limits from the CSV. Only the tested dt values can appear, so the true
boundary lies between a marker and the next larger dt of the grid.

**What to expect.** Markers at or just below their solver's line.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for solver in ["Euler", "Verlet", "RK4"]:
    style = STYLE[solver]
    sub = stab[stab["solver"] == solver]
    for k, grp in sub.groupby("k"):
        unstable = grp[grp["max_energy_ratio"] > 1 + ENERGY_GAIN_TOL]["dt"]
        below = grp[grp["dt"] < unstable.min()]["dt"] if not unstable.empty else grp["dt"]
        if below.empty:
            continue
        ax.plot(k, below.max(), marker=style["marker"], color=style["color"], ms=8, ls="none",
                mfc=style["color"] if not unstable.empty else "none")
lim = stab.drop_duplicates("k").sort_values("k")
ax.plot(lim["k"], lim["dt_limit_euler_verlet"], color="k", ls="--", lw=1, label="ω·dt = 2  (Euler, Verlet)")
ax.plot(lim["k"], lim["dt_limit_rk4"], color="k", ls="-.", lw=1, label="ω·dt = 2√2  (RK4)")
for solver in ["Euler", "Verlet", "RK4"]:
    ax.plot([], [], marker=STYLE[solver]["marker"], color=STYLE[solver]["color"], ls="none", label=solver)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Stiffness k  (N/m)")
ax.set_ylabel("Largest stable dt  (s)")
ax.set_title("Stability boundary: measured (markers) vs theory (lines)")
ax.legend(fontsize=8)
save(fig, "plot_stability_boundary")

## Energy over time

**How it is made.** `|E(t) − E₀| / E₀` along the run, sub-sampled, for the dt values recorded by
`main.cpp`.

**What to expect.** A step at each contact, as the damping dissipates energy (the same for every
solver when dt is small); in flight, Euler drifts and Verlet and RK4 stay flat. On a log scale of an
absolute value, sharp dips are points where `E(t) − E₀` changes sign.

In [ ]:
fig = time_series(ede)
fig.suptitle("Energy drift over time", y=1.0)
save(fig, "plot_energy_time_series")

## PDF report

In [ ]:
export_pdf()